# Manual2AR：无模型会话终止诊断

选择 T4 后 Run all。只记录真实遥测、系统内存与 GPU 状态，不安装或加载 SAM/Cutie，不开 HTTP 服务或隧道。
先在当前 Workspace 运行任务「Colab: 无模型诊断实验」，与本笔记本使用同一份诊断代码。参数在 configs/colab.local.json 的 diagnostics 中设置。

默认观察 30 分钟，采集进程在后台运行，Notebook 单元会返回；不会执行伪计算、模拟点击或保活循环。
请保持实验条件一致：不要在同一运行时启动其他工作，也不要中途手动删除它。另行记录启动途径（网页或 CLI）。
Google Drive 会请求本人授权。运行时被删除后重新挂载 Drive，查看每次实验目录的 latest.json、observations.jsonl 和 end.json。
缺少 end.json 只说明没有记录到正常结束，不能单独证明平台杀进程；断电、Drive 写入失败等也可能导致记录缺失。


In [ ]:
SUITE = "session_lifecycle"  # @param ["session_lifecycle"]
DURATION_MINUTES = 30  # @param {type:"integer"}
SAMPLE_SECONDS = 15  # @param {type:"integer"}
DRIVE_LOG_ROOT = "/content/drive/MyDrive/Manual2AR/logs"  # @param {type:"string"}


In [ ]:
import base64,io,os,pathlib,sys,zipfile,json
from google.colab import drive
drive.mount('/content/drive')
ROOT=pathlib.Path('/content/Manual2ARDiagnostics')
ROOT.mkdir(parents=True,exist_ok=True)
PAYLOAD="UEsDBBQAAAAIAMwQZV0AAAAAAgAAAAAAAAASAAAAZGVwbG95L19faW5pdF9fLnB5AwBQSwMEFAAAAAgAzBBlXdCJH8FiAQAAkwIAABYAAABkZXBsb3kvY29tbW9uL2ZpbGVzLnB5bVI9T8QwDN37K6Iul4iqOjEwgLoyMMBJsJ9C61Jz+VKccMe/J24pdyA8RLH9/OznpK7rXX41SJMIET90AtF7N+Jbjjqhd+KIafI5CTgFT+jehBZBx4TaiIfnp0cx+D5bcKmt67pCG3xM4p28W++eqjF6W4rSZPBVfId3xV0hCWwY0UBVVQOM6xx7ZpFc1ogPbTKo20oUW9MJTmlJM7Adsg0kZ2ADjnKEvaYesbvXhqBBN5Qhu2ul/nQ50/zuUiIdDznn1B2fbRHOSu1hwCgXh7qXyB1PSGnvD7OnZoZxaJy20K3qShnxXYYII566mZERV5t20xTG7qLHQpHi5zINm6d27CfrB1mYt/5mu1U/OX6kGTD4AI4Bm+NGCU2CUgRtzyxsS6w9RkywbEz9lx9NpkmqO+alT9fLNV7EOC+VuhwtQjC6B8mCmnllyxLQaWMuVMwrZZBqszPoDtIi8b86L+8LUEsDBBQAAAAIAMwQZV3niIKHLgYAAHMOAAAbAAAAZGVwbG95L2NvbGFiL2Vudmlyb25tZW50LnB5jVdtb9s2EP6eX8F5HyghltI0KLC6EIa0ddYMXWI0ToHNNgzaomwuMqmRlBM3yH/fHUnZcuJ20webFu+Nz909R3c6nRvLZM50npRippneJEqWG/JBlWxGDLd11SVC5rzi8CEtUQWxS06ksnym1B0ZbOxSSbLm2ggl006nc1RotSIVs0uwSMSqUtqSAfw8Cuu/jZLNWplmpXmzMvWs0mrOzXbPbMzR0VHOCzJfijKfcrkWWskVBBTFvSMCT+sVycBsGl6kc1VtotjJFEqTO76B85CIDv4cfrq+GpwPP9EuCb8+Xf/Rx19fL78Mb88/T/tXX2mw/8xHWqkqAltdcqUkj5+HMAoGb6/e315c9L/0P9IJhEVPqZPUgKuWbYXmeEoWYjE1fA4iJtJK2eAf8QQLiKN/TU4I9eLmRMiCay7nPC3VnJUpIuw9Wb3Zxe/FwQrugyjLTYR2U81ZPrX8AdCMt9IhSKiAkkdedQQnnuzDyCoxteqOS9qlyyIsYyIKIoyQBopr3minC24Rs7hrrI4JlB1pmQ0YPsx5ZcmFKPmVsheqlnlfa6V7IZpoTyy6vnG73a+srLlfDjeVX8WNDmYo4Kt5zuY2wqN2CcLYJXdS3csG8cxl0yO2RpsmO5QSPN6eHhzWuSG8NHx/y9kCeW+uEew9h5mOrmtb1ZbcC7tc8jLvBXBqzSx0FrEce4FpAd1ZS8wYm5V8Mg6JxqQ4F5gW76uHx8zwAxJclQwS4Ta6dAQFef5h2P84oR5PFIK60DyF5os0jX4VcfSeM8312BzH45tjaAtNx6ctza7TitsFvdUfz6AWRufJXyz59ip5Ozkez7CvDmiHvNRyWqrFgucR0wvT5AbKZ9Gko4NUBcdekZmCVlAerfmSSQmgu2qCKBicHvkJBEtSMFHWmneRrYijNQGp4g8C9FTOHVm56MHZXm+5tz+TG8kqs4S9GQd4OTEVu5dCLpwHx0SwYpas2IYEhJ2HvcSlwdgVB44kJbeEQVcyaQRS1e8311cYbN4EC7DolZDMcufFtVBZtizt1+uh6mwq7mCF9jQTUKJfamnFyjdKRC8lFIbIyZZHntVeOD899glxDiBd00BKDj+gIxgIFlqbVcAtC1+WWM1b0VTBFAHKgOSDE5UDlBmtbZH8AozBDMrt+sIKW3IwXtCxzLKMPDrfTwTWTc07XtQChoAT7pKirM0yG+qa72gMjKb3WljuhfY3nEIUt6y5uQNud0MoHbiwfV3O7/PM1yaQd3ZgGnW3tn7wGJtD/WZtH5eDPtZ7zrVuv78Zfry+Hf4vo67b8ewv0YU3mGiT0VCm8GZWF0Z849np7vR7wwIfJJVSSMcpTUQ+9t6LiJxc1jAs/jrIsPELRZ9BrwCXjIzSw4ncOtomFHUO7z/PKz7Y8hBgc457JmxLIkyU98zwvltC3e8fslHcNugz+98xjI/rue2bArTLcnPYuoc3nZfKNA5E4WJvtQZQBd4moA/Sv5WQ0bbBWoM8NVUJBQ8QmSgeJWdvepPWbH9JAgVtOgyZiOckckT5iK6f4pRc1GXpOpQ8Nu6exvIRY3miDY9XUF8wNPZaonWJAcK99IQGJGhQFHnuLD19DRUGuII6fBJgSknul/DhrqHUwHVDA8uT2sAEBfGz4//gboAM7owp1LqFGl6RnwAtgKJ+oL0fYUCHS6DJ7d1W839qocElI59R2YdzAiqVyslvg1ucW6idhjFa+cvwjhHTRVUncJr1yUzIE79NmwhxKvlXKUBtgL5bd02rVGnalrbkmritHQPWaxDz4sjB4Kde73aDn3p9wIdDYTd5RwgZf+Dz2uLVAuZ1ssKpXYkKv8IowmWS5MKgTAJ7Sbj7J/Mln9/R73MVqFlgUW4pMp2OXMwxmKvXWfYqfZu+oZOGM2hTJ8IA5hbKMfzRWDEJRappq5bbJwCz9RptIug+1IA6rLHQ/DvDeR6ieJEqGk8OniEENvAl7tIfYnIF3Cr5ENzzwHwgGFwyPwBTZ/dnp0sA13eBGenOUw9jhiQFxON3pPPSDDOGezON3BTmuhr1XsNfkIxEZ11y+hqjQLvtv02r2lhssfaxaOclGAGID5huB0O7d/EehgWzdy9snf7oX1BLAwQUAAAACADMEGVdrC1jS1ENAABmJQAAFgAAAGRlcGxveS9jb2xhYi93b3JrZXIucHm1Wllz3DYSftev4MoPIGMOR3Kuzaj4YDtO4lofKlvxwypTLIjEaLBDEgxASpq49N+3uwGS4MzIR2qXrtKQINho9PH1AR8fHz/j+eZaq64ugueq5FdBo0XDNW+lqgMOo7+e/x7cKr0ROjBdI/SNNPDqLKhV0Aqer2V9HRSiEXUh6lwKkxwdva3LbWBa3nYm+Y9xdGS9EhqmiCRXdS1yXCApVc5LN0eLQNzla15fiyK4le06aNciWCvTJkfnL38O1G0ttFnLJpAmuBFarmQ/8ZWsu7vgSql2DsvqNpDATCtbYCe4EisFpE2rmgZ55QHwXBbJ0fHx8ZGsGgXTub6GPRvRP6+5WZfyqn9E/vp7ZY5WWlVBw1ucErjhc3jsp+iBjhG5Fq0ZHuV1zcvhqbtqtMqFGd9vh9tWVM1KlgOlVlbDfadLWDnR4s9OmPbo6KgQK1CbvOGtyJDXEJmLgxtediJaHAVw4UiKTNK76Az/JrBnEFNSbQqpQ/tg0gvdiVjcSdNmakNPEVFYFXHNK5H2nMFnBu9DMJiVvEuJIs54zBIWA8XUW8OSaPXWcoOXMskqX1eqCIHyifrh5CQa3pFScUKhwLBwArtlUcANqFELXo1U8MItJ0UHrNCOYzsnFrXptMi4yaVMf+GlEbFEI23TJ9GZnZOsys6sw+gM1zLbOg/7cdhgrcIo8tkFxyh5LkLcZExitIKRoNTS2xmJGSdFSVeXst6ElTQGbG8UaK8zUn/mrHUbNrJw+pqICl3JKm/F5vjR/CPMvEdbb1kEfPEia8VdG8K9aUrZhixi8Wl0ebpM7PO4D7kicpcnyzRl/2YLsM9O18EbVYthjhv7yGAVtoA/MUPfYgviwbEAxjoHUKhFOdfg36qa4xzYyw5HIFHZhFE8URlejFyVLYid05+W9zQDIEA0bRC+ff9Ca6XjD6hSe/sStHdHt9GEbStMXsobEfaidGJ004CzcnxFcLQn+/7mkna9jNK0H3ELIIJkCELF7iog01q1exy4xcn0aN6j4AUAZlDxmiPGEQ4Ft2jVDUe6gJGBbA0iXc8fQIhBvE0Gq0CwblIwx2vRNtey2GPcF+O5pfJKqU3XkOj2uQLuieY/0h1KC82lEcG7rkb0oa9D5ijaTzxMBhOveJuv2ejqwONGlmVzvctibIEwef/y14sX715/IcfoOdbhlA4ylJXGWBF+exItfPP+hCoGEsQibCoxpRBNmJxGvSgOmtHX7OhfL1+9Gt3t4V01HIDfs1HizVoauo/DcHjkXdmmaOgeNLhvCPlKxQsTjtjue99EtL8AqL1R7S8Y7CeSdYt4dh5q8OXeh+DWog8NnqG/ipR4xIE5s9nBzFksZABjVGfxx/vI2f57oAtag8/L0qYXK6kNRD0FRh/kvEbFlbyrwUV4UItbsH3INnIR8FUL2Ue7hrjPbd4w2IFWpUBTuGT2S6FZzNoOEowSbhwBuNMiV7qAt0sfVgeHpj2hR4VI0InNbe8ziQt7GOWRyCQuW4JeZgTyoUfBFowyFFEg2y3ggzdy3wcM3kiiEcO+u1bEQSXatSpS9uuLC+bUBUyu5LWvHzti5uNGfO6dDf6ZTtOK5J39Ddm6bZvFfH765MfkBP6dLthjwPTQUgX7h5QEUOex5WgQbsFbnl6xj/cMvcrxmbLzt+8vWCAAewi7x/lr4BegJP3InnYwV8u/yEZACs8EpBA6YI/7FUEKWas2oga/Y89V3YIrzi62DYqMNxDvcvp0Tvu7j93a9sdul/KLnf3CIyUb8BwjNMB2UkhKHuL+W0pHtDCNqo1Y7MbOwTPDfkqvQ8L6QYslKBGyT+OUR6klZNOl2oK1QTqeiPpGalVXsMc+16S4kXkvxj1ZjZMRQxpWqmsWRQlti3GbQMHYyCxRSsdENDmnuchQnN8WKXEJC6V7S0JAN22BMgKC+7F954KpQmt/ofcXP7/9/SKmFCADX89cnPM85+/hDH11iQJYpnsx3mb+mGWdHfDLT9InupEP2ESsR+yu9iFzX40FLCeyHMK/6NX4Mw49x5EvV70WBSDgIVgGOzaqhNBFyWy+xpSeXtBk1aBHGF+aFi9n7o0DgzOCnpTZIhDAjNHnNiwgarkUO/jmG4gvmnvxsVY1ySsgEn7uCgTpq2HsazAR7APZGEDRrn/Y4NgVRLZSZGbNn3z/A1u4vV3ujANquDf7M3deLGO30fsDFYyTyCgs30HYJOXG0JZlVolZFjKz7lpZQui4XcscUmr4rpB8ZioJMH4o7XqjgjcfXv788imaGo6DL2POAdFhXOgroT/+5ERxx6sGyjxrGLuLTFA4tWMUP73xKIAA7UrghIYyAFnDV5CzPdknCGLMqbcABAd9DPyw5WTG3tc5CBQmJV0DkUeEhwm4SftrA4tc1pmsZSt5CYoCWzMP8vHA7IHqo155AS8KjdkyVhziBnEI0hmUOLU7rLejtcwoDUl2+Oo3gz2QlA0BmMVoRuk/T56cxoC+GSEBu266DIPuvIR4WaM12t1Gn3K7T9uHfTsSIBAbZeJhmrd5sHZ6cclEza9KMNAlbR9dwNWPuY3Zc5o3f70lJARnkCZD2Iqm5f0hd6AviGQF2WwrirOgg0nYMPrwPniuCmHhNWi52dg80/cTZDkd8ddGY8c15tDA8qFtjGmLe0WEJl6/nHQM3JtLdvH23fPfst/evn4BRoXZkxU/eIUGDLYiHL/ciG1KY34IyGA0jKBuP6s4xinfzb1pMxze7gRF0v4Wkpg6dQuDtcxu4LP5lazn9hXzVWiHEuoDmdCWzHZd6+XADIMSGX6mynoUPB1KV7B8asZBpo+KMRwc4sPrwKxVV1p70MLVA1T2ertIJkStLDKAfon+kFnmsFcT2lsPnlBHU5bcxsn2LCUL2BAqD6YBDwqTNr2AP55U+xiAqR76XAWmVxpPl64ESb20byxLLr0txGz2DMLcrMM/FfypeN3x8gnX2eidkN6yafBjs5l1U0xR9Bdl/EB/hrnODJxt/CryTBeS8bJdp5Om0E7VffrkZMdPsbNkN5Y0qizBaKBeQy0jmYNBDVvLfY0HlkZuDAbRQI01kKIcdrIOxmDH4FAPsbkdgQzmCnS2mXzgyt8X9APQtfAr/+/34rSlhLFrfHKh7YbLEsHggSCN+yEDmDTEkRJYXb5plKzBzLvaozMs7m3l5nR+y3XVgaptxbGDXInhgLu8u5Ol5Hob7lsjZvEUAYaC1Q8D3RWUSCOM22cMzVMUx93bd1M9A2LAqg5I8lJ1xaoEbyrYrjUgATt5wJEpJbz2qzCIrloKyGSp8jRQel5DadNdQVZczcf1vNtirgVEPUjZ5yU4s4HIAvEUXdKfNCvxlGDGq+KH71hsGYv2+HEM2870ifrRb0vj5TzMthisdSa3GmzXdlzYrrHSvIn3990Jcn7HhtezmM1QE/F+2Y3xnt7Xasa7VtnkwI84eH3WTZ1y3Aa+0FMvaPaek/pS2FuE2oEQpRIDCQkkuXrQ5yWf/XUy+2m2fPxHAt486ugP1DGLD8l4bGoJZMmkzLXj/dzU2x+tvnC2Tg8JtSzDk0MQQao6jAqeMTtv+ISACtkHNioRed1ngJ9Owz7XYoo/TjEfLcTtLfYS7sXBDgmE41ZBRckcal6OI8v7T7L1QFFGQdFrU/XPX1p9Tffy5aUYs8E1Q8cGuFnQSdWUmBaVgj2AyRg82hgSJNAHtgBUPZvu6T66n7RInwGIDHEC2yVkbKMDPQrOQZsQmiC1x4xz1ZW94LCzEgdXXetQeI5ZOxhCLmQDqI85LxQNdGyKmhkznAKriTK1pX3YbhthTTxKsgxPkbLsMVsEtulmX8RjYe80V7ehJRPTiZbXRcHr73VS8Jq2WPf6qsvF53qoeP1P+qhfZ6IrEIXfSqWfh7oGJNSspQbiIfHHbkYFsiJqVtSXs+9OTk4Wy0k2eEAaY+/ZkwlhiGse2U6Mawk6q3egfXx8/BzSY3s0hEm0NXDEGHGl1CawJ3BnDnBMIKtKFBKWLrekPDJCBHkQaIIn3rT4g62jv2cpkE/fSNVN2ku7fmabc3iADPsJ+y9IQBsoJHqv3yyJ7Y01uCly7LVtlsOpzYSeswKg6qCJqhcqP/DGnvB4+umtObJlTngwM8Isbu9L5xBRtHNOtBs8DuSpBzJKGzOAZwAWL5vaS1+Hk6PhuMpjaTgGiaanYrj9YYmd9t5gkDtB7SnEsxU5bev1KjBZGAickc3jAY77fxaYvW2Rct+O7Q+THmhyDsjctzhtS7j3mb6n6w1ZSfZ9S+dhI+wDCvQV+4IOOe+jB09iDvVAe8pff3yz26pkkNJ2iEKDkP8PDcres7wEczwJuzRbA7m3yLsWJbJXYk40YV3dsq9byjSR3n6F2FsRzUNLtTBWQXgLh/9uosGr0v7/1SRP9XWHtnNO4yH+/xO8SSA7yrh7F0LZWkHRCwLL1woc0qSXAy9oR+Dvh7/rGUXwJlyLsZyQEFG9AIKHGqn7nH6QgHHFE/bv8TGhfZJj4ZNjCIDE8mE7QGTT4+yjI5jexwqYmmUoiSxjCyuRo/8CUEsDBBQAAAAIAMwQZV2KPUwcywgAAEQXAAAZAAAAZGVwbG95L2NvbGFiL3JldGVudGlvbi5weaVYbW/cNhL+7l8hGDAoNVqtm0tyxTo6oEWdNrhLG9S+D3c+Q6ClWZtniVRJau1NkP9+M3yRtGs7TXr+4BVfZjiaeeaZoQ4PD9+DNsLY5EoNsoEmaQS/lspYUSetujYJl01ioYUOrN6eJFIlQq5Bg6whUTq5Beh5KzaQ1LxtTXF4eHggul5pm3B93XNt4GCtVZc03IIVHSRhNY5z+vdBSYhi/zVKxmdlvHDP7U0rrqLsexzGLUZcS96Oo+Gq16oGY8aZ7fhobzTwRsjrcQKPjs/DIJqDYCr0rdoWteo6JYu1aMHEk3stNmh4RUbmcWDh3nrJAuRGaCU7kDaK1DeibarZQq6h4XWUuFP6FnTc7FyZh1eoRIPbhd3mZHdurOordScB7Xzz9h+nZ2XKWj7I+gZ0gbFiObtSyhqreR/GBvRG1BBG1/1Q7c7YQUpo/eAg8X/MWG4HU9ArkgrLtR36quZ4TpwkTRhAvmyBa4kererBCljCBg32ou1M44Spxe6O+cr82Ozg4KCBdTLYOs1WGuyg5YiYQqq7NKKmwC1ZIYxaK91xm0ZJw7u+hVSjQ7KVM4QeS0KOnzyh86Akz7qJJfORWATng0G3IKbDK3/8lHktYIbWlh8Zt2zlzMvZ0JM16Npaycaw1bpVaIk7ii1J3dLvYFlBxzm8pFlh+lbg78XxZTa6yvuLwojBZ6u5DgTyEu3DeC01JqXqlnHbnlqrRZ/uqxzfiq0+St7ByiEtdU4orsGmNJllCboxoUfM8mSC1wSlETQs+/TJHdFBp/S2/OhHJN8K6eTn1uMu5A31mAtou0lDlOjvFrZ5lW94O0BJiwXSCKaBUDJlK5aN+8SatnpL30F3riwnTOHj9xsuWn7Vkrlnd7x/owG9v/KmXqDQM1b9XfzALkshbepOmodjFukL5oVwq39wa40wt6UyBblvszYeUTtStKPi0YzqamvR9ZclTRfr6sqtfBNGa2TgD+DEkWMnP2CWlROhFXqQ6QVDHsGkWZhO4LstFr8PoLcL2klRy72NxWCgic+W3JJjerbiAycnFrjbyfqUKWuzyaW6wbBgoC93cbP3V/MeUxEqNdh+sOW5HpC/MZThCTGOK+XLKUTRHXTkZYn/0WUN7okopRjSrE/xWjVQlscJtAYSNsjRf8wphPsaepukv56daq10PvPNuT/69L4XyK5Zwk0CtGc1M6ByM2iG3faQukFWVBX5rapC8BzReJlAJL2vjy7EeQNIVdJ5ccYq8+kZw+TucS5yMhsU3W0jdIrQJkL0DoR7PKlSt24U8VQr3ZRzOhvzLOapKwYTbIwadI3MRqRGW+b5IpUN68iZFZU2jAGqmubMtsOcu8V8RDZDY4dJfgec9Hcn7E0UVT1geuor5pyP4QXerR5giYBeBglKHsdX1Qj/+R9aS/N/e/7q+bcvXqy8xsIA3KY0v/DzxONugWiFyCLNHmhyCJ1tS4No0QAhLmWDXS++Y7mDhCmZxvrPkep2Nc2r/TyqS0Z9EnOuDqU9pT25C9akI6A3gHeOUIowvphFuHEEXspq1W89Wo2rPBek+zO4dXHYtUldEWG7Qay2WYgRfzxESGfr+qZTTRpcReiQCivJsXp1fDy9SFi+08JCSqqLZuh6JEH3IjlIQxTBTS1E+YZjKmfP2H+wpsdIrdvB3IQwzfupPafitLGh+nrV2W6S0lRIUqLG/UzMkdrJBW2OlWdAZX8qY6np6qEpx96xOKUOJs1OcIy2lawZtNtbYb+IOWqh8WxFdqEBHanCQott6tDla01ldnK6xMBQl5F4bTPe9Mp9d1uxZ+i7oGO0iSCTTmyAq64UepHi7O1Pb385z6fR+elv77JVGPsf2puPNnpVrt+jF6ZOC/tfZZUU9R+GCyMQOyUWVFSzFml0UogFW4WHnMUwTf3TGLjPlaKdP9arVtRb7A56bgw2NtUM/5VUlW+cqvG2wkI7d4fNOSREt6unaS5w8JOV4FGacC7yjSUilvp4JXd63Nz3kF75BT1fum7bAr6Gn0W4UY88HcWIBncA+ukBwZy6H1ybKOb/MjBEMFrm0r5aY2FGoOfMHVARM7HVY/z05THcDeiXvrSGjgu6gZQBUN+8Ol6k++BdBEju9I6j6GtsOSiBQlrdcewDUVs64nDcmWWrK0zN24e92h+D4+ncAdk85nHPAYQQ+v30hKavjyDQPZ/t6/8ynO0WsRVlW7xwkYcrj1vQ3g9IUKjg2kQGLv/6/Pjx2xjWZaPaTazesVMJ8u6CgnZTK4iW+6ISr4W/0LcD5xbU5lVGuQtGc+zyCfW0SC2PBiw2RBpW4XWF2h705bJBL8Py3fZH+mWuSyKhcKl5Yhepo7ZuRvGaC+xmfxskgdK5LWXhowt9JUBsKQv+W4uG3wfsXxOOuTvYG6Wxz2mSTqEs/v6k1DWSVTgp8DFshBrMV19jp28JadThnRwDyLKxHHR9+ZmrNwJlTTMpO/rXUXfUnB/9fPTu6OzfDKv+gj2jzyoF/XuBrd4N3F+svrsM5XGqwBSIJVlfEGks3Zl/olf2bVBMCP8e7tvGXueDU1N43OeZ+TXrvdt7gdftAu6hHiyhDq9LP9CdyV2cOkfK4RsR3k+KMfvctYoMYDklkG8qdi/ii8U8z/L9PCMFkXf86gTmB5Xy0m2PJdXtjr3O3j2uvmvchQC7s0354IsUsoO/lpXoGnpEEp+75Oz8x1//ib2ES3EJd2gAVlkM3OT7iKKLCUGX5f53rNSdXPSiyU4eIbHPIjce8Egj8vUU6N7ky6trYJr91cB8VBtiwrsPnrqMnz6L7/X1QB5+7+axa/QbCt5gexTW0hEzVEAdg+WBCZq5hx+X3EHTUwqeOnZCmpPEYR4uIeW3Lz977Ai6B4LE8Se41ZRB1P2QsNml35evyU+miEa8Lv9yfBxp9tuwGM55TWpXQSF4En0rUUg0yazVS2KfuRzfLEQQLwhOn0sC9zS/KuzYkc8PxiCjubGbKUtWVRTvqsIO1sX94H9QSwMEFAAAAAgAzBBlXWb5CVZ1AgAAoAUAABsAAABkZXBsb3kvY29sYWIvZGlhZ25vc3RpY3MucHmVVMFu2zAMvfsriOwgu3O9dsMubn0YhhYoMAwD1u5SBIZi0bEWWTIkOV1Q9N9HyWmaLu2A+hBEIs3H9/jo2Wx2KbX0mENvBKrj1iKCkHypjfOyATdS0IHruEUBiw18Nx4XxqyAawFfv10Vs9kskf1grIffzuiktaaHgftOyQVsAz/omEwRgYMym6IxfW900UpF1bdZg5Vr7rF+KlNY9Ki9NPoxx3lufW2xMVagzUffJEkisAWHzlFerWSLzaZRmFpjfO7Qe6mXLldm6bIyAXr2cWLWB/bE+JgQ/OiKEGT5PeOelYSSZjmLWrCSHUAxitFbMRb6I0CWR6jwsKisq5XhAgUrL7lymLNhXCjZ1H7UGtXuVoyWB751L/VIyrPykcHtYWz+kEUUgdS5jrHquUCTCIF7vn2p+l+9qdyaqxGrN1M3C4d2Hbm/THmnSBDlFfpPTFi5d3izMG8ddOQ80SfPjVZPKpC53sEXIYCD5j1tQBQCOrQIdx1GWyrsg0f1klbCeArt7c8ZaFzTTRAiLoWDaOzG6FYui+TnzdX1xU/S+lDY8uDqYWt1xUfddK/7W4ZtmK6LJfp0O70MqD2QGibQ0nLpEH4FmhfWGpuyG73S5k4f7D+bhGmNhRVuCCrvZCh0mx5Kn5/mpx9PspxQedCmdmRFLSjwOf90cpLNy50NqE+/GTDdDZKKzzOQbtuoBwIMf5U5r54lnVedfKH/K01DkwLYe0qaeg6yVPdHR3EFGGq+UMGQ13YktxEEOZarXY97nvqn+62jguZV+JpF9bOz8Fv0KyFtOtAXUntXxdL4Rzpfm1U8PbPVJP7tHlBUeD5/aZ7JX1BLAQIUAxQAAAAIAMwQZV0AAAAAAgAAAAAAAAASAAAAAAAAAAAAAACAAQAAAABkZXBsb3kvX19pbml0X18ucHlQSwECFAMUAAAACADMEGVd0IkfwWIBAACTAgAAFgAAAAAAAAAAAAAAgAEyAAAAZGVwbG95L2NvbW1vbi9maWxlcy5weVBLAQIUAxQAAAAIAMwQZV3niIKHLgYAAHMOAAAbAAAAAAAAAAAAAACAAcgBAABkZXBsb3kvY29sYWIvZW52aXJvbm1lbnQucHlQSwECFAMUAAAACADMEGVdrC1jS1ENAABmJQAAFgAAAAAAAAAAAAAAgAEvCAAAZGVwbG95L2NvbGFiL3dvcmtlci5weVBLAQIUAxQAAAAIAMwQZV2KPUwcywgAAEQXAAAZAAAAAAAAAAAAAACAAbQVAABkZXBsb3kvY29sYWIvcmV0ZW50aW9uLnB5UEsBAhQDFAAAAAgAzBBlXWb5CVZ1AgAAoAUAABsAAAAAAAAAAAAAAIABth4AAGRlcGxveS9jb2xhYi9kaWFnbm9zdGljcy5weVBLBQYAAAAABgAGAKEBAABkIQAAAAA="
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:archive.extractall(ROOT)
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
from deploy.colab.diagnostics import launch
settings={'suite':SUITE,'duration_minutes':DURATION_MINUTES,'sample_seconds':SAMPLE_SECONDS}
result=launch(ROOT,settings,{'enabled':True,'path':DRIVE_LOG_ROOT,'interval_seconds':SAMPLE_SECONDS})
print(json.dumps(result,ensure_ascii=False,indent=2))
print('观察已在后台开始；本单元返回不是测试结束。请查看 Drive 中该实验目录。')


## 查看结果与解释

下方单元可手动再次运行查看最新状态；Run all 不会等待 30 分钟。
`end.json: duration_completed` 表示记录器达到预定时长；`signal_15` 表示记录到 SIGTERM，不能据此判断发起者。
运行时突然消失时，最多可能丢失最后一个采集周期以及 Drive 尚未落盘的数据，平台不一定提供终止原因。
把本实验与加载模型的业务会话对比，可缩小模型/显存问题与运行时平台策略的范围，不是保活方案。
观察完成不会自动释放 GPU，请在 Colab 删除运行时，或通过 Workspace 的「Colab: 释放诊断运行时」任务释放 CLI 创建的诊断实例。
后续诊断通过 deploy/colab/diagnostics.py 的 SUITES 增加命名诊断项，复用同一份留存与导出流程。


In [ ]:
print((ROOT/'retention-status.json').read_text())
